In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 모델 이름 설정
model_name = "thisischloe/dialect-Translater"

# 토크나이저와 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import pandas as pd
import string

test_data = pd.read_csv('/content/drive/MyDrive/DACOS_NLP/test_data2.csv', encoding='ISO-8859-1')

In [39]:
# test data 크기 = 전체 데이터의 10% (약 300여개 랜덤 추출)
test_sample = test_data.sample(n=300, random_state = 30)

In [40]:
test_sample.head()

,source,target
47,We use a on a roll every day in our house.,We use a continuously succeeding every day in ...
154,french fries is very popular in this country.,chips is very popular in this country.
26,I love office with tomato sauce.,I love seat with tomato sauce.
318,Can you bring me a mail from the store?,Can you bring me a post from the store?
142,Can you bring me a the American presidential e...,Can you bring me a the U.S. presidential elect...


In [41]:
# 번역 생성 함수 정의
def preprocess(sentence):
    ## 사전 처리 요구사항
    # Tokenization
    tokens = sentence.split()
    # Case Normalization
    tokens = [token.lower() for token in tokens]
    # Punctuation Removal
    tokens = [token.translate(str.maketrans('', '', string.punctuation)) for token in tokens]
    return tokens

def word_level_accuracy(test_data, model, tokenizer):
    total_words = 0
    correct_words = 0

    for _, row in test_data.iterrows():
        source = row['source']
        target = row['target']

        source_tokens = preprocess(source)
        target_tokens = preprocess(target)

        # 예측 생성
        inputs = tokenizer(source, return_tensors="pt", truncation=True)
        outputs = model.generate(**inputs)
        predicted = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predicted_tokens = preprocess(predicted)

        # 토큰화
        total_words += len(target_tokens)
        correct_words += sum([1 for t, p in zip(target_tokens, predicted_tokens) if t == p])

    # Accuracy 평가지표
    accuracy = correct_words / total_words if total_words > 0 else 0
    return accuracy

In [42]:
# word-level accuracy 결과
accuracy = word_level_accuracy(test_sample, model, tokenizer)

print(f"Word-Level Accuracy: {accuracy:.2%}")

Word-Level Accuracy: 60.40%


In [43]:
# 번역 성공 결과만 표시하는 함수
def display_successful_translations(test_data, model, tokenizer, num_samples=10):
    successful_samples = 0

    for _, row in test_data.iterrows():
        if successful_samples >= num_samples:  # 지정된 성공 샘플 수만큼 표시
            break

        source = row['source']
        target = row['target']

        # 모델 번역 생성
        inputs = tokenizer(source, return_tensors="pt", truncation=True)
        outputs = model.generate(**inputs)
        predicted = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # 번역 성공 여부 확인
        if predicted == target:
            # 결과 출력
            print(f"Source: {source}")
            print(f"Target: {target}")
            print(f"Predicted: {predicted}")
            print("-" * 50)

            successful_samples += 1

# 성공한 번역 결과 확인
display_successful_translations(test_data, model, tokenizer, num_samples=10)

Source: I love it's very dark outside with tomato sauce.
Target: I love it's pitch black outside with tomato sauce.
Predicted: I love it's pitch black outside with tomato sauce.
--------------------------------------------------
Source: resume is very popular in this country.
Target: CV is very popular in this country.
Predicted: CV is very popular in this country.
--------------------------------------------------
Source: The french fries was left on the table.
Target: The chips was left on the table.
Predicted: The chips was left on the table.
--------------------------------------------------
Source: Can you bring me a the American presidential election from the store?
Target: Can you bring me a the U.S. presidential election from the store?
Predicted: Can you bring me a the U.S. presidential election from the store?
--------------------------------------------------
Source: Can you bring me a sneakers from the store?
Target: Can you bring me a trainers from the store?
Predicted: Ca

In [61]:
!apt-get install git
!git config --global user.name "miji0"
!git config --global user.email "kmj25b@sookmyung.ac.kr"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.11).
0 upgraded, 0 newly installed, 0 to remove and 49 not upgraded.


In [62]:
!git clone https://github.com/miji0/DialectTranslater.git
%cd DialectTranslater

Cloning into 'DialectTranslater'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 12 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 68.23 KiB | 1.31 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/DialectTranslater/DialectTranslater/DialectTranslater/DialectTranslater


In [65]:
!ls "/content/drive/MyDrive/Colab Notebooks/"

 DACON			 dialect_ipynb의_사본.ipynb  '머신러닝 hw2.ipynb'
 DACON_estate		 Untitled		     '성능평가_Word level acc.ipnyb'
 DACOS_nlp_words.ipynb	'머신러닝 hw1.ipynb'	      통분실_nlp


In [66]:
!mv "/content/drive/MyDrive/Colab Notebooks/성능평가_Word level acc.ipynb" "/content/DialectTranslater/"

mv: cannot stat '/content/drive/MyDrive/Colab Notebooks/성능평가_Word level acc.ipynb': No such file or directory
